# 3.4 · 特征缩放 / Feature Scaling

> **课程定位 / Where this fits**
> 第 4 课，**Part 3 · EDA 与数据预处理**。
> Lesson 4, **Part 3 · EDA & Preprocessing**.
>
> 很多模型对特征的**量纲**敏感：KNN(5.3)、SVM(5.5)、线性/逻辑回归带正则、PCA(6.8)、神经网络都假设各特征"可比"。若一个特征范围是 0–100000、另一个是 0–1，前者会**主导一切**。特征缩放就是把各特征拉到可比的尺度。
> Many models are sensitive to feature **scale**: KNN (5.3), SVM (5.5), regularized linear/logistic regression, PCA (6.8), and neural nets all assume features are comparable. If one feature ranges 0–100000 and another 0–1, the former **dominates everything**. Scaling brings features to a comparable scale.
>
> 💼 **实战/面试视角**："哪些模型需要缩放 / 标准化和归一化的区别 / 有异常值用哪个" 是高频题。
> 💼 **Practical/interview angle:** "which models need scaling / standardization vs normalization / which one with outliers" are common questions.

> 💡 **面试相关 / Interview-relevant**
> - "哪些模型需要缩放，哪些不需要"（出镜率 ★★★★★，树不需要）
> - "标准化 vs 归一化(MinMax) 的区别与选择"（★★★★★）
> - "有异常值时用哪种缩放（RobustScaler）"（★★★★）
> - "缩放会不会泄漏 / 在哪一步做"（★★★★★，只 fit 训练集）

---

## 学习目标 / Learning Objectives

1. 理解为什么有的模型需要缩放、有的（树）不需要。
   Understand why some models need scaling and others (trees) don't.
2. 掌握 **Standard / MinMax / Robust / MaxAbs** 的区别与适用。
   Master Standard / MinMax / Robust / MaxAbs and when to use each.
3. 知道异常值会怎样破坏 MinMax，该改用 Robust。
   Know how outliers break MinMax and to switch to Robust.
4. 了解 **QuantileTransformer** 这种非线性变换。
   Know the nonlinear QuantileTransformer.
5. **防泄漏**：缩放器只 fit 训练集。
   **Prevent leakage:** fit the scaler on the training set only.

## 目录 / TOC
1. [先建直觉 + 谁需要缩放 ⭐](#1)
2. [🍷 数据 + 缩放对 KNN 的影响 ⭐](#2)
3. [四种缩放器对比 ⭐](#3)
4. [异常值：MinMax vs Robust ⭐](#4)
5. [非线性：QuantileTransformer](#5)
6. [防泄漏：只 fit 训练集 ⭐](#6)
7. [小结](#7)


<a id="1"></a>
## 1. 先建直觉 + 谁需要缩放 ⭐ / Intuition & Who Needs It

想象用 KNN 找最近邻，特征是"年收入(元)"和"年龄(岁)"。年收入相差几万很常见，年龄相差几十岁就到头了。算欧氏距离时，**收入那一项的平方差会把年龄完全淹没**——模型几乎只看收入。把两者缩放到同一尺度后，它们才能**公平地**贡献距离。
Imagine KNN finding neighbors with features "annual income (¥)" and "age (years)". Income easily differs by tens of thousands; age by at most decades. In the Euclidean distance, **income's squared difference completely drowns out age** — the model essentially ignores age. Only after scaling to a common scale do they contribute **fairly**.

**谁需要缩放**（面试必背）：
**Who needs scaling** (memorize for interviews):
- ✅ **需要**：距离/内积类（KNN、SVM、K-Means）、带正则的线性模型（正则惩罚 $\|\mathbf{w}\|$ 对量纲敏感）、PCA（看方差）、神经网络（帮助梯度下降）。
  ✅ **Needs it:** distance/inner-product methods (KNN, SVM, K-Means), regularized linear models (the penalty on $\|\mathbf{w}\|$ is scale-sensitive), PCA (variance-based), neural nets (helps gradient descent).
- ❌ **不需要**：树类模型（决策树、随机森林、GBDT/XGBoost）——它们只比较"某特征是否 ≤ 阈值"，与量纲无关。
  ❌ **Doesn't:** tree models (decision tree, random forest, GBDT/XGBoost) — they only compare "is feature ≤ threshold", independent of scale.


<a id="2"></a>
## 2. 数据 + 缩放对 KNN 的影响 ⭐ / Data & Impact on KNN

用 **Wine**（13 个化学成分特征，量纲差异巨大：如 proline 上千、而某些比值在 0–5）。直接看缩放对 KNN 和随机森林的不同影响。
Use **Wine** (13 chemical features with wildly different scales: proline in the thousands, some ratios in 0–5). We compare scaling's effect on KNN vs Random Forest.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_wine
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 140)
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(42)

wine = load_wine(as_frame=True)
X, y = wine.data, wine.target
print(f"Wine: {X.shape}, {len(np.unique(y))} 类 classes")
print("\n各特征量级 feature scales (注意差异巨大 huge disparity):")
print(X.agg(["min","max","mean","std"]).T.round(2).head(8))


In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline

# 不缩放 vs 缩放(用 Pipeline 把缩放和模型打包, 自动只 fit 训练折, 防泄漏, 见 3.12)
acc_raw = cross_val_score(KNeighborsClassifier(), X, y, cv=5).mean()
acc_scaled = cross_val_score(make_pipeline(StandardScaler(), KNeighborsClassifier()), X, y, cv=5).mean()
print(f"KNN 不缩放 unscaled: {acc_raw:.1%}")
print(f"KNN 缩放后 scaled:   {acc_scaled:.1%}  (提升 +{(acc_scaled-acc_raw)*100:.0f} 个百分点, 仅因缩放!)")

# 对照: 树模型对缩放免疫 / trees are immune to scaling
acc_rf_raw = cross_val_score(RandomForestClassifier(random_state=0), X, y, cv=5).mean()
acc_rf_scaled = cross_val_score(make_pipeline(StandardScaler(), RandomForestClassifier(random_state=0)), X, y, cv=5).mean()
print(f"\n随机森林 RF 不缩放: {acc_rf_raw:.1%}  缩放后: {acc_rf_scaled:.1%}  (几乎不变, 树对缩放免疫)")


<a id="3"></a>
## 3. 四种缩放器对比 ⭐ / Four Scalers Compared

四种最常用的缩放器，**都是线性变换**（只改位置和尺度，不改分布形状）：
The four most common scalers, **all linear transforms** (they shift/rescale but don't change the distribution's shape):

| 缩放器 scaler | 公式 formula | 输出范围 | 适用 |
|---|---|---|---|
| **StandardScaler** | $(x-\mu)/\sigma$ | 均值0、std1（无界）| 默认首选；数据近正态 |
| **MinMaxScaler** | $(x-\min)/(\max-\min)$ | $[0,1]$ | 需要固定范围（如图像像素、神经网络）|
| **RobustScaler** | $(x-\text{median})/\text{IQR}$ | 无界 | **有异常值**时（用中位数/IQR，抗异常）|
| **MaxAbsScaler** | $x/\max\|x\|$ | $[-1,1]$ | **稀疏数据**（不破坏 0，不平移）|


In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, MaxAbsScaler

col = X["proline"].values.reshape(-1, 1)    # 拿量纲最大的 proline 演示 / reshape 成列(n,1) 供 sklearn
scalers = {
    "原始 raw": col.ravel(),
    "Standard": StandardScaler().fit_transform(col).ravel(),     # 减均值除std
    "MinMax":   MinMaxScaler().fit_transform(col).ravel(),       # 压到 [0,1]
    "Robust":   RobustScaler().fit_transform(col).ravel(),       # 减中位数除IQR
    "MaxAbs":   MaxAbsScaler().fit_transform(col).ravel(),       # 除以最大绝对值
}
fig, axes = plt.subplots(1, 5, figsize=(15, 2.8))
for ax, (name, v) in zip(axes, scalers.items()):
    ax.hist(v, bins=30); ax.set_title(f"{name}\n[{v.min():.1f}, {v.max():.1f}]", fontsize=9)
plt.suptitle("proline 经各 scaler 后: 形状不变, 只是位置/尺度变了(都是线性变换)", y=1.08)
plt.tight_layout(); plt.show()
print("Standard/MinMax/Robust/MaxAbs 都是线性变换 → 分布形状不变, 只改位置和尺度")


<a id="4"></a>
## 4. 异常值：MinMax vs Robust ⭐ / Outliers: MinMax vs Robust

**MinMax 对异常值极度脆弱**：它用 min/max 定范围，一个极端大值会把所有正常数据**挤进一个极小的区间**，模型几乎分不开。**RobustScaler** 改用中位数和 IQR（都对异常值稳健），正常数据保持正常分布，异常值留在远处不挤压主体。**有异常值就别用 MinMax，改用 Robust。**
**MinMax is extremely fragile to outliers:** it uses min/max for the range, so one huge value crams all normal data **into a tiny interval**, leaving them nearly indistinguishable. **RobustScaler** uses the median and IQR (both robust), keeping normal data well-spread while the outlier sits far away without squashing the bulk. **With outliers, drop MinMax for Robust.**


In [ ]:
data = np.append(rng.normal(50, 10, 200), [500]).reshape(-1, 1)   # 200 正常 + 1 个 500 的异常
mm = MinMaxScaler().fit_transform(data).ravel()
rb = RobustScaler().fit_transform(data).ravel()

fig, axes = plt.subplots(1, 2, figsize=(11, 3))
axes[0].hist(mm[:-1], bins=40); axes[0].axvline(mm[-1], color="r", label="异常值 outlier")
axes[0].set_title(f"MinMax: 正常数据全挤在 [0, {mm[:-1].max():.2f}]\n异常值霸占整个 [0,1]"); axes[0].legend()
axes[1].hist(rb[:-1], bins=40); axes[1].axvline(rb[-1], color="r", label="异常值 outlier")
axes[1].set_title(f"Robust: 正常数据分布正常\n异常值 z={rb[-1]:.0f} 留在远处不挤压主体"); axes[1].legend()
plt.tight_layout(); plt.show()
print("MinMax 下正常数据被压到极小区间, 模型几乎分不开 → 有异常值时禁用 MinMax, 改 Robust")


<a id="5"></a>
## 5. 非线性：QuantileTransformer / Nonlinear

前面四种都是线性缩放（不改形状）。**QuantileTransformer** 是**非线性**变换：它把数据按分位数强行映射成均匀分布或正态分布，能把极偏的数据"掰直"。代价是：**它改变了数据点之间的相对关系**（不再保线性），用时要清楚这一点。
The four above are linear (shape-preserving). **QuantileTransformer** is **nonlinear**: it maps data by quantiles onto a uniform or normal distribution, forcibly straightening heavily-skewed data. The cost: **it changes the relative relationships between points** (no longer linear) — use it knowingly.


In [ ]:
from sklearn.preprocessing import QuantileTransformer
import scipy.stats as st

skewed = rng.lognormal(0, 1, 1000).reshape(-1, 1)        # 极右偏的对数正态数据
# output_distribution='normal': 把数据按分位映射成正态分布 / map to a normal distribution
normalized = QuantileTransformer(output_distribution="normal", random_state=0).fit_transform(skewed).ravel()

fig, axes = plt.subplots(1, 2, figsize=(11, 3))
axes[0].hist(skewed, bins=50); axes[0].set_title(f"原始 lognormal (skew={st.skew(skewed.ravel()):.2f})")
axes[1].hist(normalized, bins=50); axes[1].set_title(f"Quantile→normal (skew={st.skew(normalized):.2f})")
plt.tight_layout(); plt.show()
print("强行正态化, 但记住这是非线性变换, 改变了数据点之间的相对关系(慎用)")


<a id="6"></a>
## 6. 防泄漏：只 fit 训练集 ⭐ / Prevent Leakage

和填补一样的纪律：缩放器的参数（均值/std、min/max、中位数/IQR）**只能从训练集学**，再用同一组参数去 transform 测试集。如果在全数据上 fit，测试集的统计量就泄漏进了训练。
Same discipline as imputation: the scaler's parameters (mean/std, min/max, median/IQR) must be **learned from the training set only**, then applied to the test set. Fitting on all data leaks test statistics into training.

> 一个易混点：正确做法下，**测试集缩放后的均值不会恰好等于 0**（因为用的是训练集的 μ）——这恰恰是"没泄漏"的标志，不是 bug。
> A common confusion: done right, the **test set's scaled mean won't be exactly 0** (it uses the train μ) — that's the *sign* of no leakage, not a bug.


In [ ]:
from sklearn.model_selection import train_test_split
X_tr, X_te = train_test_split(X[["proline"]], test_size=0.3, random_state=0)

# ❌ 错: 在全数据(含 test)上 fit / WRONG: fit on all data
wrong_scaler = StandardScaler().fit(X[["proline"]])
# ✅ 对: fit 只看 train / RIGHT: fit on train only
right_scaler = StandardScaler().fit(X_tr)

print(f"全数据 proline 均值/std (错误用): {wrong_scaler.mean_[0]:.1f} / {wrong_scaler.scale_[0]:.1f}")
print(f"仅 train proline 均值/std (正确):  {right_scaler.mean_[0]:.1f} / {right_scaler.scale_[0]:.1f}")
X_te_scaled = right_scaler.transform(X_te)   # test 用 train 学到的 (μ,σ) 来变换
print(f"\ntest 缩放后均值 = {X_te_scaled.mean():.3f} (不精确为 0! 因为用的是 train 的 μ — 这才对)")
print("💡 用 Pipeline (3.12) 把缩放和模型打包, 自动只 fit 训练折, 永不手滑")


<a id="7"></a>
## 7. 小结 / Summary

```
需要缩放: KNN/SVM/K-Means(距离), 带正则的线性模型, PCA, 神经网络
不需要: 树模型(决策树/RF/GBDT/XGBoost, 只比阈值)
四种线性缩放器(不改分布形状):
  Standard (x-μ)/σ — 默认; MinMax →[0,1] 需固定范围; Robust 中位数/IQR — 抗异常; MaxAbs →[-1,1] 稀疏数据
有异常值: 别用 MinMax(被极值挤压), 用 Robust
QuantileTransformer: 非线性, 强行正态/均匀化, 改变相对关系(慎用)
防泄漏 ⭐: 缩放器只 fit 训练集; test 缩放后均值≠0 正是没泄漏的标志; 用 Pipeline
```

### 💡 面试速查 / Interview cheat-sheet
1. **距离/正则/PCA/NN 需要缩放，树不需要**。
   Distance/regularization/PCA/NN need scaling; trees don't.
2. **Standard（均值0方差1）vs MinMax（[0,1]）**：MinMax 需固定范围但怕异常值。
   Standard (mean0/var1) vs MinMax ([0,1]); MinMax needs a fixed range but fears outliers.
3. **有异常值用 RobustScaler**（中位数/IQR）。
   Use RobustScaler (median/IQR) with outliers.
4. **稀疏数据用 MaxAbs**（不平移，保住 0）。
   Use MaxAbs for sparse data (no shift, keeps zeros).
5. **缩放器只 fit 训练集**，用 Pipeline 自动保证（防泄漏）。
   Fit the scaler on train only; Pipelines enforce it (no leakage).

### 下一节 / Next
**3.5 类别变量编码**——模型只吃数字，文字类别（性别、城市、职业）怎么变成数字：One-Hot、Ordinal、目标编码及其泄漏陷阱。
**3.5 Categorical Encoding** — models eat numbers; how to turn text categories into numbers: One-Hot, Ordinal, target encoding and its leakage trap.
